# 02 — SQL Analysis
**Reel Insights: TMDB Movie Industry EDA**

Connects to `data/movies.db` (built in `01_data_cleaning.ipynb`) using `sqlite3` and answers the 10 required SQL questions, each with a short insight.

## Setup

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/movies.db')

def q(sql):
    return pd.read_sql(sql, conn)

## Q1 — Action movies released after 2015
*Concept: JOIN (movies ⋈ movie_genres ⋈ genres)*

In [2]:
q1 = q('''
    SELECT m.title, m.release_date
    FROM movies m
    JOIN movie_genres mg ON m.movie_id = mg.movie_id
    JOIN genres g ON mg.genre_id = g.genre_id
    WHERE g.genre_name = 'Action' AND m.release_date > '2015-12-31'
    ORDER BY m.release_date
''')
print(f"{len(q1)} Action movies released after 2015")
q1.head(10)

268 Action movies released after 2015


,title,release_date
0,The 5th Wave,2016-01-14 00:00:00
1,Kung Fu Panda 3,2016-01-23 00:00:00
2,Zoolander 2,2016-02-06 00:00:00
3,Deadpool,2016-02-09 00:00:00
4,Triple 9,2016-02-19 00:00:00
5,Grimsby,2016-02-24 00:00:00
6,Gods of Egypt,2016-02-25 00:00:00
7,They Call Me Jeeg,2016-02-25 00:00:00
8,London Has Fallen,2016-03-02 00:00:00
9,Allegiant,2016-03-09 00:00:00


**Insight:** Lists every post-2015 Action release in the dataset — useful as the base filter for any genre-specific trend analysis (e.g. Action output vs. box office over time).

## Q2 — Top 10 highest-grossing movies with their genre(s)
*Concept: JOIN + ORDER BY + LIMIT*

In [3]:
q2 = q('''
    SELECT m.title, m.revenue, GROUP_CONCAT(g.genre_name, ', ') AS genres
    FROM movies m
    JOIN movie_genres mg ON m.movie_id = mg.movie_id
    JOIN genres g ON mg.genre_id = g.genre_id
    GROUP BY m.movie_id
    ORDER BY m.revenue DESC
    LIMIT 10
''')
q2

,title,revenue,genres
0,Avatar,2.923706e+09,"Action, Science Fiction, Adventure"
1,Avengers: Endgame,2.799439e+09,"Adventure, Science Fiction, Action"
2,Avatar: The Way of Water,2.334485e+09,"Action, Adventure, Science Fiction"
3,Titanic,2.264162e+09,"Drama, Romance"
4,Star Wars: The Force Awakens,2.068224e+09,"Adventure, Action, Science Fiction"
5,Avengers: Infinity War,2.052415e+09,"Adventure, Action, Science Fiction"
6,Spider-Man: No Way Home,1.921207e+09,"Action, Adventure, Science Fiction"
7,Zootopia 2,1.868209e+09,"Animation, Adventure, Comedy, Mystery, Family"
8,Inside Out 2,1.698864e+09,"Animation, Adventure, Comedy, Family"
9,Jurassic World,1.671537e+09,"Adventure, Science Fiction, Thriller"


**Insight:** The top grossers skew heavily toward Action/Adventure/Sci-Fi combinations, confirming that big-budget tentpole genres dominate box-office revenue in this dataset.

## Q3 — Average budget & revenue by genre
*Concept: JOIN + GROUP BY + AVG()*

In [4]:
q3 = q('''
    SELECT g.genre_name,
           ROUND(AVG(m.budget), 0) AS avg_budget,
           ROUND(AVG(m.revenue), 0) AS avg_revenue,
           COUNT(DISTINCT m.movie_id) AS movie_count
    FROM movies m
    JOIN movie_genres mg ON m.movie_id = mg.movie_id
    JOIN genres g ON mg.genre_id = g.genre_id
    GROUP BY g.genre_name
    ORDER BY avg_revenue DESC
''')
q3

,genre_name,avg_budget,avg_revenue,movie_count
0,Adventure,109359964.0,390214464.0,674
1,Animation,83360605.0,359743487.0,246
2,Family,84584257.0,343096752.0,344
3,Science Fiction,95259923.0,323560110.0,463
4,Action,93294567.0,300054700.0,808
5,Fantasy,86177842.0,294297494.0,396
6,Comedy,52569069.0,211351278.0,762
7,Music,41069388.0,197452578.0,49
8,Romance,34611126.0,162789647.0,349
9,Thriller,49387696.0,160515200.0,673


**Insight:** Genres like Adventure and Animation post the highest average revenue, but this raw average is skewed by unknown ($0) budget/revenue rows still included here — Notebook 03 recomputes this after excluding unknown financials for a cleaner comparison.

## Q4 — Top 10 actors by number of movie appearances
*Concept: GROUP BY + COUNT()*

In [5]:
q4 = q('''
    SELECT actor_name, COUNT(DISTINCT movie_id) AS movie_count
    FROM cast
    GROUP BY actor_name
    ORDER BY movie_count DESC
    LIMIT 10
''')
q4

,actor_name,movie_count
0,Samuel L. Jackson,39
1,Robert De Niro,38
2,Brad Pitt,38
3,Johnny Depp,37
4,Tom Hanks,33
5,Willem Dafoe,32
6,Scarlett Johansson,32
7,Mark Wahlberg,32
8,Tom Cruise,31
9,Morgan Freeman,31


**Insight:** The most frequently cast actors in this dataset are established leading names with long, prolific track records — useful signal for a "bankability" score in casting analytics.

## Q5 — Directors with more than 3 movies
*Concept: GROUP BY + HAVING*

In [6]:
q5 = q('''
    SELECT person_name AS director_name, COUNT(DISTINCT movie_id) AS movie_count
    FROM crew
    WHERE job = 'Director'
    GROUP BY person_name
    HAVING COUNT(DISTINCT movie_id) > 3
    ORDER BY movie_count DESC
''')
q5

,director_name,movie_count
0,Steven Spielberg,27
1,Tim Burton,19
2,Ridley Scott,19
3,Martin Scorsese,16
4,Robert Zemeckis,15
...,...,...
212,Andrew Adamson,4
213,Alfonso Cuarón,4
214,Alexandre Aja,4
215,Adam Wingard,4


**Insight:** A small set of directors account for a disproportionate share of the catalog — these prolific directors are good candidates for a dedicated director-performance dashboard page.

## Q6 — Top 10 most frequently used keywords
*Concept: GROUP BY + COUNT()*

In [7]:
q6 = q('''
    SELECT keyword_name, COUNT(DISTINCT movie_id) AS movie_count
    FROM movie_keywords
    GROUP BY keyword_name
    ORDER BY movie_count DESC
    LIMIT 10
''')
q6

,keyword_name,movie_count
0,based on novel or book,449
1,sequel,416
2,duringcreditsstinger,296
3,aftercreditsstinger,242
4,murder,176
5,based on true story,166
6,new york city,161
7,villain,160
8,3d animation,155
9,based on comic,154


**Insight:** The dominant keywords point to recurring thematic clusters (e.g. hero/superhero, based-on-novel adaptations) — this is exactly the metadata a recommendation engine would use for content similarity.

## Q7 — Above-average budget, below-average revenue
*Concept: Subquery comparison against AVG()*

In [8]:
q7 = q('''
    SELECT title, budget, revenue
    FROM movies
    WHERE budget > (SELECT AVG(budget) FROM movies WHERE budget > 0)
      AND revenue < (SELECT AVG(revenue) FROM movies WHERE revenue > 0)
      AND budget > 0 AND revenue > 0
    ORDER BY budget DESC
''')
print(f"{len(q7)} movies spent more than average but earned less than average")
q7.head(10)

273 movies spent more than average but earned less than average


,title,budget,revenue
0,The Marvels,274800000.0,206136825.0
1,Red One,250000000.0,185700759.0
2,Wake Up Dead Man: A Knives Out Mystery,210000000.0,4000000.0
3,Wonder Woman 1984,200000000.0,169601036.0
4,Luca,200000000.0,51074773.0
5,Green Lantern,200000000.0,219851172.0
6,Mulan,200000000.0,69965374.0
7,Onward,200000000.0,141888276.0
8,Jungle Cruise,200000000.0,220889446.0
9,The Gray Man,200000000.0,454023.0


**Insight:** These are the dataset's clearest commercial underperformers relative to spend — a natural watch-list for a 'what went wrong' case-study section, and a sanity check against any budget-based revenue-forecasting model.

## Q8 — Actors who appeared in a specific director's movies (Christopher Nolan)
*Concept: JOIN (cast ⋈ crew)*

In [9]:
director_name = 'Christopher Nolan'
q8 = q(f'''
    SELECT DISTINCT ca.actor_name
    FROM cast ca
    JOIN crew cr ON ca.movie_id = cr.movie_id
    WHERE cr.job = 'Director' AND cr.person_name = '{director_name}'
    ORDER BY ca.actor_name
''')
print(f"{len(q8)} distinct actors have worked with {director_name}")
q8.head(15)

84 distinct actors have worked with Christopher Nolan


,actor_name
0,Aaron Eckhart
1,Al Pacino
2,Alon Aboutboul
3,Andy Serkis
4,Aneurin Barnard
5,Anne Hathaway
6,Barry Keoghan
7,Benny Safdie
8,Callum Keith Rennie
9,Carrie-Anne Moss


**Insight:** This query template generalizes to any director in the crew table and is the exact logic behind the Streamlit app's 'actors who worked with director X' lookup on the Query Explorer page.

## Q9 — Genre with highest average rating (min. 20 movies)
*Concept: JOIN + GROUP BY + HAVING + AVG()*

In [10]:
q9 = q('''
    SELECT g.genre_name,
           ROUND(AVG(m.vote_average), 2) AS avg_rating,
           COUNT(DISTINCT m.movie_id) AS movie_count
    FROM movies m
    JOIN movie_genres mg ON m.movie_id = mg.movie_id
    JOIN genres g ON mg.genre_id = g.genre_id
    GROUP BY g.genre_name
    HAVING COUNT(DISTINCT m.movie_id) >= 20
    ORDER BY avg_rating DESC
''')
q9

,genre_name,avg_rating,movie_count
0,War,7.36,85
1,History,7.34,112
2,Music,7.32,49
3,Drama,7.25,937
4,Animation,7.20,246
5,Western,7.13,33
6,Romance,7.03,349
7,Crime,7.01,387
8,Family,6.94,344
9,Mystery,6.88,216


**Insight:** After filtering out niche genres with too few movies to be statistically meaningful, the top-rated genre reflects durable critical/audience favorites rather than a handful of outlier hits.

## Q10 — Top 10 movies by number of keywords tagged
*Concept: JOIN + GROUP BY*

In [11]:
q10 = q('''
    SELECT m.title, COUNT(mk.keyword_id) AS keyword_count
    FROM movies m
    JOIN movie_keywords mk ON m.movie_id = mk.movie_id
    GROUP BY m.movie_id
    ORDER BY keyword_count DESC
    LIMIT 10
''')
q10

,title,keyword_count
0,Smile 2,98
1,Taken 3,70
2,Silent Hill,63
3,Twelve Monkeys,57
4,War for the Planet of the Apes,54
5,The Shining,52
6,Dawn of the Planet of the Apes,51
7,How to Train Your Dragon,50
8,13 Hours: The Secret Soldiers of Benghazi,49
9,Pacific Rim: Uprising,49


**Insight:** Heavily-tagged movies tend to be dense, high-concept films (multiple plot threads, franchises, genre-blends) — rich keyword coverage like this is what makes a similarity-based recommendation engine effective for that title.

In [12]:
conn.close()
print("Connection closed.")

Connection closed.


## Summary & Recommendation

- All 10 required queries run successfully against the cleaned SQLite database.
- Financial aggregates (Q3, Q7) are sensitive to the unknown-budget/unknown-revenue rows flagged in Notebook 01; Notebook 03 (Pandas) repeats the profit/ROI-style analysis after explicitly excluding those rows for a cleaner comparison.
- **Recommendation:** the director/actor lookup pattern (Q8) and the top-N patterns (Q2, Q4, Q6, Q10) are reused directly as the saved queries in the Streamlit Query Explorer page.